# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - '입력 질문'에 대해 검색된 문서나 '정보의 관련성과 정확성'을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 '생성된 응답의 품질, 정확성'등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 '준비된 데이터셋을 활용'하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - '실제 사용자 트래픽과 피드백'을 기반으로 시스템의 '실시간 성능'을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `pip install ragas rapidfuzz`

## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [ ]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [ ]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [4]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )

    return splitter.split_text(olympic_text)

In [5]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):


    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    client = QdrantClient(url="http://localhost:6333")

    # 컬렉션 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
      
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)

    return vectorstore


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [6]:
vectorstore = get_vectorstore()

retriever = get_retriever(vectorstore)
retriever

VectorStoreRetriever(tags=['QdrantVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000001F3D396E660>, search_kwargs={'k': 5})

In [ ]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from operator import itemgetter

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_doc_to_str(documents:list[Document])->list[str]:
    """
    VectorStore에 조회한 문서들(list[Document])에서 내용(page_content)만 추출해서 list[str] 로 반환.
    RAGAS 평가시 context는 각 검색한 문서를 list[str] 로 받기 때문에 이렇게 처리.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        list[str]: 각 문서의 내용만 추출해서 리스트에 담는다.
    """
    return [doc.page_content for doc in documents]

# RAG 체인 -> 평가데이터셋 만드는 RAG Chain
#          -> 최종응답: LLM의 응답(str), 검색한 문서들(list[str])
chain = RunnablePassthrough() | {
    "context":retriever | format_doc_to_str,
    "query":RunnablePassthrough() 
} | { 
    "response": prompt | model | parser,  # LLM 응답
    "retrieved_context": itemgetter("context") # 검색한 문서들
}
# RunnablePassthrough() - LCEL 체인 만들려면 구성 요소 중 하나가 Runnable이어야 하는데 이 체인은 dict | dict 구조기 때문에 앞에 추가

In [8]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [9]:
print(res.keys())

dict_keys(['response', 'retrieved_context'])


In [13]:
res['response']

'기원전 776년, 그리스 올림피아에서 열렸습니다.'

In [14]:
res["retrieved_context"]

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

# RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.

> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [15]:
# 주피터노트북 환경에서 비동기적 처리 위해
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [ ]:
#############################
# testset -> Context(문서) - [질문 - 정답 답변 + (Retriever가 찾은 문서+LLM 응답: Chain 생성)]
# 1. Context(문서) 추출: TestsetGenerator -> 질문과 정답 답변 생성

import random
# 데이터셋 생성 시 사용할 Context 추출
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "olympic_info_wiki"

# 전체 저장된 문서 중에서 k개만 sampling
info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count  # 총 문서 개수 조회

results, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=total_docs
)
# 랜덤하게 k(5)개를 sampling
sample_docs:"list[PointStruct]" = random.sample(results, 5)  # 리스트에서 랜덤하게 k(5)개 추출(k: 데이터셋 개수)

# PointStruct - payload: page_content, metadatae
# page_content만 추출해서 list[str]
docs = [point.payload["page_content"] for point in sample_docs]


In [21]:
total_docs
sample_docs

[Record(id='68ab38c5-3896-4711-8336-bcc4002f4653', payload={'page_content': '올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애인이 참여하는 패럴림픽, 스

In [23]:
# 테스트 셋 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# TestsetGenerator는 gpt-5 이후 버전은 사용 불가
## Langchain의 LLM 모델과 Embedding 모델 사용 -> RAGAS에서 사용할 수 있도록 반환(Wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context=""" 
-  사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON 문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" # 질문/답변 생성 시 LLM에게 전달할 System Prompt 설정
)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\3279304446.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\3279304446.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [24]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용, 테스트셋 개수(질문-답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node 73be37fd-ae6f-4cfe-88a8-2163a6840926 does not have a summary. Skipping filtering.
Node 6c2237b2-d4c7-4a7b-ad8d-ff21f80de412 does not have a summary. Skipping filtering.
Node 1b7b4389-4775-4438-85d2-d5f924e12e26 does not have a summary. Skipping filtering.
Node 6d8b385a-f2c6-4dab-b144-aa86083b0a6c does not have a summary. Skipping filtering.
Node ffd15bef-4ca8-4ef5-9253-f015d1015282 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [25]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림픽이랑 어떻게 연결되는지 궁금한데, 혹시 고대 그리스 올림피아가 지금 올림픽의 시작점이라고 볼 수 있나요?', retrieved_contexts=None, reference_contexts=['올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 

In [44]:
sample1 = testset.samples[0].eval_sample # 10개 중 첫번째 테스트 데이터
print("사용자 질문 :", sample1.user_input)
print("Context :", sample1.reference_contexts) # 질문과 답변을 만들 때 사용한 context(검색 시 찾아야 하는 문서)
print("생성된 답변(정답) :", sample1.reference)
#### 평가 대상 RAG system 이용해 채워 넣어야 함
print("평가 대상 RAG의 답변 :", sample1.response)
print("평가 대상 RAG가 검색한 Context :", sample1.retrieved_contexts)

사용자 질문 : 고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림픽이랑 어떻게 연결되는지 궁금한데, 혹시 고대 그리스 올림피아가 지금 올림픽의 시작점이라고 볼 수 있나요?
Context : ['올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)은 전 세계 각 대륙 각국에서 모인 수천 명의 선수가 참가해 여름과 겨울에 스포츠 경기를 하는 국제적인 대회이다. 전 세계에서 가장 큰 지구촌 최대의 스포츠 축제인 올림픽은 세계에서 가장 인지도있는 국제 행사이다. 올림픽은 2년마다 하계 올림픽과 동계 올림픽이 번갈아 열리며, 국제 올림픽 위원회(IOC)가 감독하고 있다. 또한 오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다. 그리고 19세기 말에 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어, 근대 올림픽을 부활시켰다. 이를 위해 쿠베르탱 남작은 1894년에 IOC를 창설했으며, 2년 뒤인 1896년에 그리스 아테네에서 제 1회 올림픽이 열렸다. 이때부터 IOC는 올림픽 운동의 감독 기구가 되었으며, 조직과 활동은 올림픽 헌장을 따른다. 오늘날 전 세계 대부분의 국가에서 올림픽 메달은 매우 큰 영예이며, 특히 올림픽 금메달리스트는 국가 영웅급의 대우를 받으며 스포츠 스타가 된다. 국가별로 올림픽 메달리스트들에게 지급하는 포상금도 크다. 대부분의 인기있는 종목들이나 일상에서 쉽게 접하고 즐길 수 있는 생활스포츠 종목들이 올림픽이라는 한 대회에서 동시에 열리고, 전 세계 대부분의 국가 출신의 선수들이 참여하는 만큼 전 세계 스포츠 팬들이 가장 많이 시청하는 이벤트이다. 2008 베이징 올림픽의 모든 종목 누적 시청자 수만 47억 명에 달하며, 이는 인류 역사상 가장 많은 수의 인구가 시청한 이벤트였다.  \n또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예

In [46]:
# 생성된 Testset을 Pandas DataFrame으로 변환

eval_df = testset.to_pandas()
eval_df.shape

(11, 7)

In [47]:
eval_df.head(3)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림...,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",오늘날의 올림픽은 기원전 8세기부터 서기 5세기까지 고대 그리스 올림피아에서 열렸던...,Olympic Bidding Coordinator,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,서울 올림픽 언제 열렷나요?,[올림픽 개최지는 해당 올림픽 개최 7년 전에 IOC 위원들의 투표로 결정된다. 개...,1988년 하계 올림픽이 대한민국의 서울에서 열렸습니다.,Olympic Bidding Coordinator,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
2,IF는 올림픽에서 어떤 역할을 담당하고 있습니까?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구로, 예를 들어 국...",Olympic Bidding Coordinator,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer


In [49]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']  # 질문 조회
resp = chain.invoke(q)  # dic[response, retriever_context]

In [50]:
resp['response']

'네, 맞습니다.  \nContext에 따르면 **오늘날의 올림픽은 기원전 8세기부터 서기 5세기에 이르기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전에서 비롯되었다**고 되어 있습니다. 또한 **현대 올림픽은 19세기 말 피에르 드 쿠베르탱 남작이 고대 올림피아 제전에서 영감을 얻어 부활시킨 것**이라고 설명합니다.\n\n따라서 **고대 그리스 올림피아는 지금의 올림픽의 시작점이라고 볼 수 있습니다.**  \n다만 현재의 올림픽은 고대 올림피아 제전을 그대로 이어받은 것이 아니라, **그 전통에서 영감을 받아 근대에 다시 만들어진 것**입니다.'

In [51]:
resp['retrieved_context']

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

In [52]:
eval_df['user_input']

0     고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림...
1                                       서울 올림픽 언제 열렷나요?
2                           IF는 올림픽에서 어떤 역할을 담당하고 있습니까?
3                                       올림피아 뭐야? 왜 중요해?
4                                    프랑스 동계올림픽 언제 열렷나요?
5     올림픽이 고대 그리스 올림피아에서 시작된 거 맞지요? 그럼 그 고대 올림피아 제전에...
6     고대 그리스에서 올림픽 경기가 어떻게 시작되었으며, 헤라클레스와 제우스에 관한 신화...
7     고대 그리스 올림피아에서 시작한 올림픽이 오늘날 IOC가 감독하는 국제 대회로 어떻...
8     국제올림픽위원회(IOC)가 올림픽 개최 도시 선정과 종목 변경, 그리고 국가 올림픽...
9                     국제 올림픽 위원회 뭐 하는 곳이야? 올림픽에서 역할 뭐야?
10                            국가 올림픽 위원회 뭐 하는데? 몇 개 있어?
Name: user_input, dtype: str

In [53]:
# testset의 모든 데이터에 대한 llm 응답과 retriever 검색 결과 추가
response_list = [] # LLM 응답들을 저장할 리스트
retrieved_context_list = [] # retriever가 검색한 문서들을 저장할 리스트

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])

In [54]:
print(len(response_list), len(retrieved_context_list))

11 11


In [55]:
response_list

['네, **고대 그리스 올림피아에서 올림픽이 처음 열렸다는 내용은 Context상 맞습니다.**  \nContext에는 **고대 올림피아 경기가 보통 기원전 776년으로 인정된다**고 되어 있습니다.\n\n그리고 **지금의 올림픽은 이 고대 올림피아 제전에서 비롯되었다**고 Context에 명시되어 있습니다.  \n또한 **19세기 말 피에르 드 쿠베르탱이 고대 올림피아 제전에서 영감을 받아 근대 올림픽을 부활시켰다**고 되어 있어, 고대 올림피아는 **현대 올림픽의 역사적 시작점으로 연결되는 출발점**이라고 볼 수 있습니다.\n\n따라서 질문에 답하면:\n- **고대 그리스 올림피아에서 올림픽이 처음 열렸다는 것은 맞습니다.**\n- **고대 그리스 올림피아는 지금 올림픽의 시작점이라고 볼 수 있습니다.**',
 '정보가 부족해 답을 할 수없습니다.',
 '국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구입니다. 예를 들어 FIFA는 축구를, FIVB는 배구를 주관하며, 올림픽에는 현재 35개의 국제경기연맹이 각 종목을 대표합니다.',
 '올림피아는 고대 그리스에서 올림피아 제전이 열렸던 장소입니다.  \n주어진 context에 따르면, 오늘날의 올림픽은 **기원전 8세기부터 서기 5세기까지 고대 그리스 올림피아에서 열렸던 올림피아 제전**에서 비롯되었습니다.\n\n중요한 이유는 두 가지입니다.\n1. **올림픽의 기원**이 되는 장소이기 때문입니다.\n2. 올림픽 성화도 매 대회 전 **그리스의 올림피아에서 채화**되기 때문입니다.\n\n즉, 올림피아는 **고대 올림픽의 발생지이자 현대 올림픽의 상징적 시작점**이라서 중요합니다.',
 '프랑스의 동계올림픽은 **1924년 샤모니에서** 처음 열렸습니다. 또한 **동계 올림픽은 프랑스의 알베르빌에서 열린 1992년까지** 하계 올림픽과 같은 해에 개최되었습니다.',
 '네, 맞습니다. Context에 따르면 올림픽은 **고대 그리스 올림피아에서 열렸던 올림피아 제전**에서 비롯되었습니다.\

In [56]:
retrieved_context_list

[['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스

In [70]:
##########################
# eval_df에 컬럼으로 추가
##########################
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,reponse,reference_context,retrieved_contexts,response
0,고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림...,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,오늘날의 올림픽은 기원전 8세기부터 서기 5세기까지 고대 그리스 올림피아에서 열렸던...,Olympic Bidding Coordinator,MISSPELLED,LONG,single_hop_specific_query_synthesizer,"네, **고대 그리스 올림피아에서 올림픽이 처음 열렸다는 내용은 Context상 맞...",[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,"네, **고대 그리스 올림피아에서 올림픽이 처음 열렸다는 내용은 Context상 맞..."
1,서울 올림픽 언제 열렷나요?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",1988년 하계 올림픽이 대한민국의 서울에서 열렸습니다.,Olympic Bidding Coordinator,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...","[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",정보가 부족해 답을 할 수없습니다.
2,IF는 올림픽에서 어떤 역할을 담당하고 있습니까?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구로, 예를 들어 국...",Olympic Bidding Coordinator,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,"국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구입니다. 예를 들어...","[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구입니다. 예를 들어..."
3,올림피아 뭐야? 왜 중요해?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...","올림피아는 고대 그리스에서 올림픽 경기가 열린 장소로, 여러 도시 국가 대표선수들이...",Olympic Program Coordinator,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer,올림피아는 고대 그리스에서 올림피아 제전이 열렸던 장소입니다. \n주어진 cont...,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...","[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",올림피아는 고대 그리스에서 올림피아 제전이 열렸던 장소입니다. \n주어진 cont...
4,프랑스 동계올림픽 언제 열렷나요?,[동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 ...,"1회 동계올림픽은 1924년 프랑스의 샤모니에서 11일간 진행되었고, 1992년에는...",Olympic Program Coordinator,MISSPELLED,SHORT,single_hop_specific_query_synthesizer,프랑스의 동계올림픽은 **1924년 샤모니에서** 처음 열렸습니다. 또한 **동계 ...,[동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 ...,[동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 ...,프랑스의 동계올림픽은 **1924년 샤모니에서** 처음 열렸습니다. 또한 **동계 ...


In [71]:
# eval_df를 RAGAS에 평가 데이터셋 타입을 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=11)

In [72]:
####################
# 평가
####################
from ragas.metrics import (
    LLMContextRecall,  # Context Recall
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness,
    AnswerRelevancy
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\3238770186.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\3238770186.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\3238770186.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [73]:
# 평가할 때 사용할 LLM, Embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

# Metric(평가 지표) 객체를 List로 묶기 - 내가 평가할 지표들만
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)
]
# 평가 진행
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\653647804.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_18000\653647804.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Evaluating:   0%|          | 0/44 [00:00<?, ?it/s]

Exception raised in Job[2]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception raised in Job[21]: TimeoutError()
Exception raised in Job[24]: TimeoutError()
Exception raised in Job[26]: TimeoutError()
Exception raised in Job[28]: TimeoutError()


In [74]:
eval_result

{'context_recall': 1.0000, 'llm_context_precision_with_reference': 0.8389, 'faithfulness': 0.9808, 'answer_relevancy': 0.5022}

In [ ]:
# print(type(eval_result))
## 개별 평가 데이터에 대한 평가 점수 확인
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,고대 그리스 올림피아에서 올림픽이 처음 열렸다는게 맞나요? 그리고 그게 지금의 올림...,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,"네, **고대 그리스 올림피아에서 올림픽이 처음 열렸다는 내용은 Context상 맞...",오늘날의 올림픽은 기원전 8세기부터 서기 5세기까지 고대 그리스 올림피아에서 열렸던...,1.0,0.916667,NaN,0.640949
1,서울 올림픽 언제 열렷나요?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",정보가 부족해 답을 할 수없습니다.,1988년 하계 올림픽이 대한민국의 서울에서 열렸습니다.,1.0,0.333333,NaN,0.000000
2,IF는 올림픽에서 어떤 역할을 담당하고 있습니까?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구입니다. 예를 들어...","국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구로, 예를 들어 국...",1.0,1.000000,NaN,0.544315
3,올림피아 뭐야? 왜 중요해?,"[올림픽(영어: Olympic Games, 프랑스어: Jeux olympiques)...",올림피아는 고대 그리스에서 올림피아 제전이 열렸던 장소입니다. \n주어진 cont...,"올림피아는 고대 그리스에서 올림픽 경기가 열린 장소로, 여러 도시 국가 대표선수들이...",1.0,1.000000,NaN,0.641743
4,프랑스 동계올림픽 언제 열렷나요?,[동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 ...,프랑스의 동계올림픽은 **1924년 샤모니에서** 처음 열렸습니다. 또한 **동계 ...,"1회 동계올림픽은 1924년 프랑스의 샤모니에서 11일간 진행되었고, 1992년에는...",NaN,1.000000,NaN,0.534176
5,올림픽이 고대 그리스 올림피아에서 시작된 거 맞지요? 그럼 그 고대 올림피아 제전에...,[고대 올림피아 경기를 제대로 구현한 최초의 시도는 혁명 시대의 프랑스에서 1796...,"네, 맞습니다. Context에 따르면 올림픽은 **고대 그리스 올림피아에서 열렸던...",올림픽은 기원전 8세기부터 서기 5세기까지 고대 그리스 올림피아에서 열렸던 올림피아...,NaN,NaN,NaN,0.735209
6,"고대 그리스에서 올림픽 경기가 어떻게 시작되었으며, 헤라클레스와 제우스에 관한 신화...",[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,"고대 올림픽 경기(올림피아 경기)의 유래는 **수수께끼로 남아 있으며**, 여러 전...","고대 그리스에서 올림픽 경기는 여러 도시 국가의 대표선수들이 모여 육상, 격투기, ...",NaN,1.000000,NaN,0.000000
7,고대 그리스 올림피아에서 시작한 올림픽이 오늘날 IOC가 감독하는 국제 대회로 어떻...,[고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이...,고대 그리스 올림피아에서 열리던 올림피아 제전은 기원전 8세기경 시작된 범그리스 경...,올림픽은 고대 그리스 올림피아에서 기원전 8세기부터 서기 5세기까지 열렸던 올림피아...,NaN,0.583333,1.000000,0.569059
8,"국제올림픽위원회(IOC)가 올림픽 개최 도시 선정과 종목 변경, 그리고 국가 올림픽...","[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...","IOC는 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종...","국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시...",1.0,0.805556,0.923077,0.719214
9,국제 올림픽 위원회 뭐 하는 곳이야? 올림픽에서 역할 뭐야?,"[올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기...",국제 올림픽 위원회(IOC)는 **모든 올림픽 활동을 통솔하는 단체**입니다. \...,국제 올림픽 위원회(IOC)는 올림픽을 감독하는 기관이야. IOC는 올림픽 개최 도...,1.0,0.916667,1.000000,0.614897
